# Decisions, loops and functions: sizing a water pipe

*Python for Engineers: Case Studies from Calculator to Simulation* · Notebook 02 · Yanwei Wang

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/python-for-engineers-case-studies/blob/main/notebooks/02_Decisions_Loops_and_Functions.ipynb)

Notebook 01 ran every line exactly once. Real engineering calculations need more: they **decide** (is this flow laminar or turbulent?), they **repeat** (check every standard pipe size, or step a draining tank forward in time), and they **reuse** a calculation many times. Python's `if`, `for`, `while` and `def` provide these three abilities.

The engineering question: water must flow at 5 litres per second through a long straight pipe, and the pressure drop must stay below 300 Pa per metre of pipe. **What is the smallest standard pipe that meets this limit?** We will answer it twice: first by trying sizes one by one, then by bisection, which finds the exact diameter.

**How to study:** predict each output before running a cell. This notebook still uses only plain Python.

<!-- pfe-study-guide -->
## Study guide

| | |
|---|---|
| **Status** | Core · Part 1, First programs |
| **Time** | about 40 min (about 70 min with the practice) |
| **Before this** | Notebook 01. |
| **Essential** | All sections. |
| **Optional on a first reading** | — |
| **Next** | 03, From a data file to a result. |

**You have understood this notebook if you can, without looking back:**

1. Classify a pipe flow as laminar, transitional or turbulent with `if`/`elif`/`else`.
2. Accumulate a total or a count with a `for` loop.
3. Simulate a draining tank with a `while` loop, and explain how the time step affects the answer.
4. Write a function with parameters and a return value, and check it with `assert`.
5. Choose a pipe diameter by guess-and-check with `break` and `continue`, then refine it by bisection.

**Not expected of you:** Deriving the Darcy–Weisbach or Blasius formulas.

*The comment on every code line is a reading aid. In your own programs, comment the reasons and the non-obvious steps.*

## 1. Decisions with `if`, `elif` and `else`

Whether water flows smoothly (**laminar**) or chaotically (**turbulent**) depends on the **Reynolds number**

$$\mathrm{Re} = \frac{\rho v D}{\mu},$$

where $v$ is the mean velocity, $D$ the pipe diameter and $\mu$ the viscosity. In a pipe, flow is laminar below about 2300 and turbulent above about 4000. In between it is **transitional**: it may switch unpredictably.

An `if` statement runs its indented block only when its condition is `True`. `elif` ("else if") is tested only when the conditions above it were `False`, and `else` catches everything left. **Indentation matters**: the four spaces show which lines belong to which branch.

In [1]:
# Load the math module for π and square roots.
import math
# Density of water at about 20 °C, in kg/m³.
rho = 998.0
# Dynamic viscosity of water at about 20 °C, in Pa·s.
mu = 1.0e-3
# Mean flow velocity, in m/s (a slow trickle).
velocity = 0.02
# Pipe inner diameter, in m.
diameter = 0.05
# Reynolds number (dimensionless).
reynolds = rho * velocity * diameter / mu
# Laminar if below 2300 ...
if reynolds < 2300:
    # ... this line runs only when the condition above is True.
    regime = "laminar"
# ... otherwise, transitional if at most 4000 ...
elif reynolds <= 4000:
    # ... runs only when Re is between 2300 and 4000.
    regime = "transitional"
# ... and turbulent in every remaining case.
else:
    # Runs only when both conditions above were False.
    regime = "turbulent"
# Report the result.
print(f"Re = {reynolds:.0f} → {regime}")

Re = 998 → laminar


> **Try it:** change `velocity` to `0.1` and then to `1.0`, and predict the regime before you run the cell.

## 2. Repetition with `for`

A `for` loop repeats its indented block once for every item in a sequence. `range(n)` produces the whole numbers 0, 1, …, n − 1. A list, written in square brackets, can hold any values you choose.

Let us print a small table of Reynolds numbers for several velocities in the same 5 cm pipe.

In [2]:
# Velocities to examine, in m/s: a list of five floats.
velocities = [0.02, 0.05, 0.08, 0.2, 1.0]
# Table header.
print(" v (m/s)        Re   regime")
# The loop variable v takes each value of the list in turn.
for v in velocities:
    # Reynolds number for this velocity.
    re_v = rho * v * diameter / mu
    # Same classification as before, now inside the loop.
    if re_v < 2300:
        # Laminar range.
        label = "laminar"
    # Between the two limits.
    elif re_v <= 4000:
        # Transitional range.
        label = "transitional"
    # Everything else.
    else:
        # Turbulent range.
        label = "turbulent"
    # One table row per velocity; :>8 right-aligns in 8 characters.
    print(f"{v:>8.2f} {re_v:>9.0f}   {label}")

 v (m/s)        Re   regime
    0.02       998   laminar
    0.05      2495   transitional
    0.08      3992   transitional
    0.20      9980   turbulent
    1.00     49900   turbulent


A loop often **accumulates** a result: start a running total at zero and add to it each time round. Here a pipeline is built from sections of different lengths; the loop adds up the total length and counts the sections longer than 10 m.

In [3]:
# Lengths of the pipeline sections, in m.
section_lengths = [12.0, 6.5, 6.5, 18.0, 3.2, 9.8]
# Running total of length, starts at zero.
total_length = 0.0
# Counter of long sections, starts at zero.
long_sections = 0
# Visit each section once.
for length in section_lengths:
    # Add this section to the running total.
    total_length += length
    # Is this a long section?
    if length > 10:
        # Count it.
        long_sections += 1
# Report the results.
print(f"total length = {total_length} m, sections longer than 10 m: {long_sections}")
# Check against Python's built-in sum().
assert math.isclose(total_length, sum(section_lengths))

total length = 56.0 m, sections longer than 10 m: 2


## 3. Repetition with `while`: draining the tank step by step

A `for` loop is used when you know in advance how many times to repeat. A `while` loop repeats **as long as a condition stays `True`**, which suits questions such as "how long until the tank is empty?".

Notebook 01 used a formula for the drain time of a water tank: 2.0 m diameter, 3.5 m deep, with a 5 cm hole. Now we check that formula a completely different way, by **simulation**. Every second, water leaves through the hole at the Torricelli rate $C_d A_\text{hole}\sqrt{2gh}$, so the level drops by

$$\Delta h = \frac{C_d A_\text{hole}\sqrt{2gh}}{A_\text{tank}}\,\Delta t .$$

We repeat this small step until the level reaches zero. If the two methods agree, both are probably right.

In [4]:
# Gravitational acceleration, in m/s².
g = 9.81
# Cross-sectional area of the tank (2.0 m diameter), in m².
area_tank = math.pi * (2.0 / 2) ** 2
# Area of the 5 cm drain hole, in m².
area_hole = math.pi * (0.05 / 2) ** 2
# Discharge coefficient of the hole.
c_d = 0.61
# Starting water level, in m.
level = 3.5
# Time step of the simulation, in s.
dt = 1.0
# Elapsed time, starts at zero.
time = 0.0
# Keep stepping as long as water remains in the tank.
while level > 0:
    # Outflow rate at the current level, in m³/s.
    outflow = c_d * area_hole * math.sqrt(2 * g * level)
    # Lower the level by the volume lost in one time step.
    level -= outflow * dt / area_tank
    # Advance the clock by one step.
    time += dt
# The formula from Notebook 01, for comparison.
t_formula = area_tank / (c_d * area_hole) * math.sqrt(2 * 3.5 / g)
# Report both results in minutes.
print(f"simulated: {time / 60:.2f} min   formula: {t_formula / 60:.2f} min")
# They should agree to within a few time steps.
assert abs(time - t_formula) < 5 * dt

simulated: 36.87 min   formula: 36.93 min


The simulation lands within a few seconds of the formula. Two independent methods give the same answer, which is strong evidence that both are correct.

> **Danger: infinite loops.** If the condition of a `while` loop never becomes `False`, the loop never ends. Had we written `level += …` instead of `level -= …`, the level would rise forever. In Jupyter, use *Kernel → Interrupt* to stop a runaway cell. A safe habit is to also count the steps and stop after a generous maximum, as the bisection loop in Section 6 does.

## 4. Functions: write a calculation once, use it many times

Our next task needs the pressure drop along a pipe for many different diameters. Copying the formula each time invites mistakes. A **function** packages a calculation under a name. `def` starts the definition, the names in parentheses are its **parameters** (inputs), and `return` sends the result back to whoever called it.

For turbulent flow in a smooth pipe, the **Darcy–Weisbach** equation with the **Blasius** friction factor gives the pressure drop per metre:

$$\frac{\Delta p}{L} = f\,\frac{\rho v^2}{2D}, \qquad f = 0.316\,\mathrm{Re}^{-1/4}.$$

Blasius's formula is reliable only for roughly $4000 < \mathrm{Re} < 10^5$. A careful function reports when it is used outside that range, and that check is itself a decision inside the function.

In [5]:
# Define a function: name, parameters, then an indented body.
def pressure_gradient(flow_rate, d):
    # A docstring says what the function does and what it returns.
    """Return (pressure drop per metre in Pa/m, Reynolds number) for water in a smooth pipe."""
    # Pipe cross-section, π d²/4, in m².
    area = math.pi * d ** 2 / 4
    # Mean velocity = flow rate / area, in m/s.
    v = flow_rate / area
    # Reynolds number for this pipe.
    re = rho * v * d / mu
    # Blasius friction factor (valid for about 4000 < Re < 100000).
    f = 0.316 * re ** -0.25
    # Darcy–Weisbach pressure drop per unit length.
    dp_per_m = f * rho * v ** 2 / (2 * d)
    # Send both results back to the caller as a pair.
    return dp_per_m, re

# Required flow: 5 litres per second, converted to m³/s.
Q = 5.0 / 1000
# Call the function for a 10 cm pipe and unpack the two results.
dp, re_check = pressure_gradient(Q, 0.10)
# Report them.
print(f"D = 0.100 m: Δp/L = {dp:.1f} Pa/m at Re = {re_check:,.0f}")

D = 0.100 m: Δp/L = 40.3 Pa/m at Re = 63,535


Variables created inside a function, such as `area` and `v`, are **local**: they exist only while the function runs, so they cannot clash with names elsewhere. The function can still *read* `rho` and `mu`, which were defined outside it.

## 5. Guess-and-check: the smallest standard pipe that works

Pipes come in standard sizes. We try them from smallest to largest and stop at the first one that meets the limit. Two loop keywords help:

- `continue` skips the rest of the current pass and moves on to the next item;
- `break` leaves the loop immediately.

Sizes for which the Blasius formula is not valid are skipped, rather than trusted.

In [6]:
# Standard inner diameters to try, smallest first, in m.
standard_sizes = [0.050, 0.065, 0.080, 0.100]
# Limit on the pressure drop, in Pa per metre of pipe.
dp_limit = 300.0
# No size has been chosen yet: None means "no value".
chosen = None
# Try each standard size in turn.
for d in standard_sizes:
    # Pressure gradient and Reynolds number for this size.
    dp_d, re_d = pressure_gradient(Q, d)
    # Outside the Blasius range, the result cannot be trusted ...
    if re_d > 1e5:
        # ... so report it and skip to the next size.
        print(f"D = {d:.3f} m: Re = {re_d:,.0f} is outside the Blasius range, skipped")
        # Go straight to the next diameter.
        continue
    # Report this size.
    print(f"D = {d:.3f} m: Δp/L = {dp_d:6.1f} Pa/m")
    # The first size within the limit is the answer ...
    if dp_d <= dp_limit:
        # ... remember it ...
        chosen = d
        # ... and stop searching: larger pipes are also fine, but cost more.
        break
# Report the decision.
print(f"smallest suitable standard pipe: D = {chosen} m")
# The search must have found something.
assert chosen is not None

D = 0.050 m: Re = 127,069 is outside the Blasius range, skipped
D = 0.065 m: Δp/L =  311.5 Pa/m
D = 0.080 m: Δp/L =  116.2 Pa/m
smallest suitable standard pipe: D = 0.08 m


## 6. Bisection: find the exact diameter

The 65 mm pipe is slightly too narrow and the 80 mm pipe is comfortably wide enough. Which diameter between them gives **exactly** 300 Pa/m? Formulas like ours cannot easily be rearranged for $D$. **Bisection** finds it anyway:

1. Start with an interval $[D_\text{low}, D_\text{high}]$ that contains the answer: too much pressure drop at the low end, too little at the high end.
2. Test the midpoint. If it gives too much pressure drop, the answer lies in the upper half; otherwise it lies in the lower half.
3. Keep the half that contains the answer, and repeat until the interval is as narrow as you need.

Each step halves the interval, so 20 steps shrink it about a million times.

In [7]:
# Lower end: the 65 mm pipe gives too much pressure drop.
d_low = 0.065
# Upper end: the 80 mm pipe gives too little.
d_high = 0.080
# Stop when the interval is narrower than 1 micrometre.
tolerance = 1e-6
# Count the halvings.
steps = 0
# Safety limit, so a mistake can never cause an infinite loop.
max_steps = 100
# Repeat while the interval is still too wide, within the safety limit.
while d_high - d_low > tolerance and steps < max_steps:
    # Midpoint of the current interval.
    d_mid = (d_low + d_high) / 2
    # Pressure gradient at the midpoint (the Reynolds number is not needed here).
    dp_mid, _ = pressure_gradient(Q, d_mid)
    # Too much pressure drop: the pipe must be wider ...
    if dp_mid > dp_limit:
        # ... so the answer is in the upper half.
        d_low = d_mid
    # Otherwise the pipe is wide enough ...
    else:
        # ... so the answer is in the lower half.
        d_high = d_mid
    # One more halving done.
    steps += 1
# The midpoint of the final interval is our answer.
d_exact = (d_low + d_high) / 2
# Report it in millimetres.
print(f"Δp/L = 300 Pa/m at D = {d_exact * 1000:.3f} mm after {steps} halvings")
# Check: the pressure gradient there should be very close to the limit.
assert math.isclose(pressure_gradient(Q, d_exact)[0], dp_limit, rel_tol=1e-4)

Δp/L = 300 Pa/m at D = 65.517 mm after 14 halvings


**An independent check.** With the Blasius friction factor, the pressure gradient for a fixed flow rate is proportional to $D^{-4.75}$. (Substitute $v = 4Q/\pi D^2$ into the formulas above and collect the powers of $D$.) Starting from the 80 mm value, we can therefore solve for the exact diameter directly:

$$D = D_{80}\left(\frac{(\Delta p/L)_{80}}{300}\right)^{1/4.75}.$$

In [8]:
# Pressure gradient of the 80 mm pipe, used as the reference point.
dp_80, _ = pressure_gradient(Q, 0.080)
# Exact diameter from the power law Δp/L ∝ D^(-4.75).
d_power_law = 0.080 * (dp_80 / dp_limit) ** (1 / 4.75)
# Compare the two methods, in millimetres.
print(f"bisection: {d_exact * 1000:.3f} mm   power law: {d_power_law * 1000:.3f} mm")
# They agree to within the bisection tolerance.
assert abs(d_exact - d_power_law) < 2 * tolerance

bisection: 65.517 mm   power law: 65.517 mm


## 7. Common loop mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| `range(1, 10)` when you wanted 10 items | the last item is missing (it gives 1 … 9) | remember that `range` stops *before* its end value |
| forgetting to update the loop variable in a `while` | the cell never finishes | check that every pass moves towards the stopping condition; add a maximum step count |
| wrong indentation | a line runs once instead of every pass, or the reverse | indentation *is* the structure in Python; check it line by line |
| changing a running total inside the wrong block | totals that are too small or too large | print the total inside the loop for a few passes to watch it grow |

## 8. Practice

Try each question in the empty cell, then compare with the worked answers.

1. Write a `for` loop that prints the regime for velocities 0.01, 0.05 and 0.5 m/s in a 2 cm pipe.
2. Repeat the tank simulation with `dt = 10` s. How much does the simulated drain time change? Why does a smaller step agree better with the formula?
3. The client tightens the limit to 100 Pa/m (still 5 L/s). Which standard size from Section 5 is needed now?
4. Write a function `reynolds_number(v, d)` that returns $\mathrm{Re}$, and use it to check your answer to question 1.

In [9]:
# Your work for the practice questions goes here.

**Worked answers.** Run the cell after you have tried the questions.

In [10]:
# Q4 first, because Q1 can use it: a function for the Reynolds number.
def reynolds_number(v, d):
    # Return Re for water at about 20 °C.
    return rho * v * d / mu

# Q1: regimes in a 2 cm pipe.
for v in [0.01, 0.05, 0.5]:
    # Reynolds number from the new function.
    re_q = reynolds_number(v, 0.02)
    # A compact three-way decision.
    if re_q < 2300:
        # Laminar.
        name = "laminar"
    # Transitional band.
    elif re_q <= 4000:
        # Transitional.
        name = "transitional"
    # Otherwise turbulent.
    else:
        # Turbulent.
        name = "turbulent"
    # Report the case.
    print(f"Q1: v = {v} m/s, Re = {re_q:.0f}, {name}")
# Q2: the same draining simulation with a ten-second step.
level, time = 3.5, 0.0
# Step until the tank is empty.
while level > 0:
    # Lower the level by one 10-second step of outflow.
    level -= c_d * area_hole * math.sqrt(2 * g * level) * 10.0 / area_tank
    # Advance the clock.
    time += 10.0
# Each step uses the outflow at the start of the step, when the level is highest, so coarse steps drain too fast.
print(f"Q2: dt = 10 s gives {time / 60:.2f} min vs formula {t_formula / 60:.2f} min")
# Q3: the first standard size within a tighter 100 Pa/m limit at 5 L/s.
for d in standard_sizes:
    # Pressure gradient and Reynolds number at 5 L/s.
    dp_q, re_q = pressure_gradient(Q, d)
    # Use only sizes where Blasius is valid and the tighter limit is met.
    if re_q <= 1e5 and dp_q <= 100.0:
        # Report the first suitable size ...
        print(f"Q3: D = {d} m gives {dp_q:.1f} Pa/m (Re = {re_q:,.0f})")
        # ... and stop.
        break

Q1: v = 0.01 m/s, Re = 200, laminar
Q1: v = 0.05 m/s, Re = 998, laminar
Q1: v = 0.5 m/s, Re = 9980, turbulent
Q2: dt = 10 s gives 36.50 min vs formula 36.93 min
Q3: D = 0.1 m gives 40.3 Pa/m (Re = 63,535)


## What to take away

- `if` / `elif` / `else` choose between alternatives; the first `True` condition wins.
- `for` repeats once per item; `while` repeats until a condition fails. `break` and `continue` change the flow inside a loop.
- Every `while` loop needs a guaranteed way to stop.
- A **function** gives a calculation a name, inputs and an output, so you can reuse and test it.
- **Bisection** finds where a quantity crosses a target by repeatedly halving an interval. Checking its answer by an independent method is good practice.

Notebook 03 puts these tools to work on real-looking data: reading a temperature log from a file, cleaning it and summarising it.